# QMIX (EPyMARL, VMAS Navigation) - Colab Notebook

This notebook runs **QMIX** using EPyMARL's built-in implementation on VMAS `navigation` and reproduces the same 3x2 training-chart layout used in `ippo/ippo_colab.ipynb`.

VMAS configuration is aligned to `gnn_mapp/gnn_mapp_vmas.ipynb` for fair comparison:
- `scenario="navigation"`
- `NUM_AGENTS=10`
- `MAX_CYCLES=100`
- `CONTINUOUS_ACTIONS=False`
- `TOTAL_TIMESTEPS=1_000_000`

1. Set runtime to GPU (optional).
2. Run cells top-to-bottom.
3. Update constants in one place for future sweeps.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/uoe-agents/epymarl.git"
REPO_DIR = Path("epymarl")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])

# Keep Colab's preinstalled CUDA-enabled torch/torchvision when available.
requirements = (REPO_DIR / "requirements.txt").read_text().split()
skip = {"torch", "torchvision"}
filtered = [pkg for pkg in requirements if pkg.split("==")[0] not in skip]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *filtered])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "vmas[gymnasium]", "matplotlib", "numpy"])

import torch
print(f"Dependencies ready. torch={torch.__version__}, cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import os
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# =========================
# Experiment constants (aligned with gnn_mapp_vmas)
# =========================
ALGO_CONFIG = "qmix"
EXPERIMENT_NAME = "qmix_vmas_navigation_colab"

# VMAS setup
VMAS_SCENARIO = "navigation"
VMAS_BRIDGE_MODULE = "vmas_epymarl_bridge"
VMAS_ENV_ID = "vmas-navigation-custom-v0"
ENV_KEY = f"{VMAS_BRIDGE_MODULE}:{VMAS_ENV_ID}"

NUM_AGENTS = 10
VMAS_NUM_ENVS = 1
MAX_CYCLES = 100
CONTINUOUS_ACTIONS = False
VMAS_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Reward aggregation to match mean-over-agents reporting in gnn_mapp_vmas
COMMON_REWARD = True
REWARD_SCALARISATION = "mean"

TOTAL_TIMESTEPS = 1_000_000
ROLLOUT_LENGTH = 2048
BATCH_SIZE = 64
NUM_EPOCHS = None  # not used by QMIX runner; kept for structural parity with gnn_mapp_vmas
LOG_EVERY = 10

SEED = 42
USE_CUDA = bool(torch.cuda.is_available())

LOG_INTERVAL = ROLLOUT_LENGTH * LOG_EVERY
TEST_INTERVAL = LOG_INTERVAL
TEST_EPISODES = 10

OUTPUT_DIR = "qmix_outputs/vmas_navigation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS_ROOT = Path("epymarl") / "results" / "sacred"
PLOT_PATH = str(Path(OUTPUT_DIR) / "qmix_vmas_training_colab.png")

# Sacred metric key preference order
POLICY_LOSS_KEYS = ["loss", "policy_loss"]
VALUE_LOSS_KEYS = ["loss", "value_loss"]
ENTROPY_KEYS = ["entropy"]  # QMIX usually has no policy entropy metric
BELLMAN_KEYS = ["td_error_abs", "loss"]
RETURN_KEYS = ["return_mean", "test_return_mean"]
EP_LENGTH_KEYS = ["ep_length_mean", "episode_length_mean", "episode_limit_mean"]

ENTROPY_FALLBACK_VALUE = 0.0
BELLMAN_FALLBACK_TO_VALUE_LOSS = True


QMIX is value-based; EPyMARL typically does not report a policy-entropy metric. In this notebook the `entropy` panel is filled with a constant `0.0` so plot structure stays identical to IPPO.
To keep rewards comparable with `gnn_mapp_vmas`, we set `common_reward=True` with `reward_scalarisation="mean"` (mean over agents).


In [ ]:
def _numeric_run_dirs(root: Path):
    if not root.exists():
        return []
    return sorted([p for p in root.iterdir() if p.is_dir() and p.name.isdigit()], key=lambda p: int(p.name))


def ensure_vmas_registration_module():
    bridge_path = Path("epymarl") / f"{VMAS_BRIDGE_MODULE}.py"
    bridge_lines = [
        "from gymnasium.envs.registration import register",
        "import vmas",
        "",
        f'ENV_ID = "{VMAS_ENV_ID}"',
        "",
        "def _make_env(**kwargs):",
        "    return vmas.make_env(",
        f'        scenario="{VMAS_SCENARIO}",',
        '        num_envs=kwargs.pop("num_envs", 1),',
        '        n_agents=kwargs.pop("n_agents", 10),',
        '        device=kwargs.pop("device", "cpu"),',
        '        continuous_actions=kwargs.pop("continuous_actions", False),',
        '        wrapper="gymnasium",',
        '        max_steps=kwargs.pop("max_steps", None),',
        '        seed=kwargs.pop("seed", None),',
        "        **kwargs,",
        "    )",
        "",
        "try:",
        '    register(id=ENV_ID, entry_point=__name__ + ":_make_env")',
        "except Exception:",
        "    pass",
        "",
    ]
    bridge_path.write_text("\n".join(bridge_lines))


def run_epymarl_experiment():
    ensure_vmas_registration_module()
    before = {p.name for p in _numeric_run_dirs(RESULTS_ROOT)}

    cmd = [
        sys.executable,
        "src/main.py",
        f"--config={ALGO_CONFIG}",
        "--env-config=gymma",
        "with",
        f"name={EXPERIMENT_NAME}",
        f"seed={SEED}",
        f"use_cuda={USE_CUDA}",
        f"t_max={TOTAL_TIMESTEPS}",
        f"test_interval={TEST_INTERVAL}",
        f"log_interval={LOG_INTERVAL}",
        f"runner_log_interval={LOG_INTERVAL}",
        f"test_nepisode={TEST_EPISODES}",
        f"common_reward={COMMON_REWARD}",
        f"reward_scalarisation={REWARD_SCALARISATION}",
        f"batch_size={BATCH_SIZE}",
        f"env_args.key={ENV_KEY}",
        f"env_args.time_limit={MAX_CYCLES}",
        f"env_args.max_steps={MAX_CYCLES}",
        f"env_args.n_agents={NUM_AGENTS}",
        f"env_args.num_envs={VMAS_NUM_ENVS}",
        f"env_args.continuous_actions={CONTINUOUS_ACTIONS}",
        f"env_args.device={VMAS_DEVICE}",
        f"env_args.seed={SEED}",
    ]

    if NUM_EPOCHS is not None:
        cmd.append(f"epochs={NUM_EPOCHS}")

    print("Running command:")
    print(" ".join(cmd))

    proc = subprocess.Popen(
        cmd,
        cwd="epymarl",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")

    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"EPyMARL run failed with exit code {rc}")

    after_dirs = _numeric_run_dirs(RESULTS_ROOT)
    created = [p for p in after_dirs if p.name not in before]
    run_dir = created[-1] if created else (after_dirs[-1] if after_dirs else None)
    if run_dir is None:
        raise FileNotFoundError("No Sacred run directory found under epymarl/results/sacred")
    return run_dir


def load_sacred_metrics(run_dir: Path):
    metrics_path = run_dir / "metrics.json"
    if not metrics_path.exists():
        raise FileNotFoundError(f"Missing metrics file: {metrics_path}")
    with metrics_path.open("r") as f:
        return json.load(f)


def pick_series(metrics, keys):
    for key in keys:
        if key in metrics and metrics[key].get("values"):
            vals = np.asarray(metrics[key]["values"], dtype=np.float32)
            return vals, key
    return np.asarray([], dtype=np.float32), None


def pad_to_length(arr, length, fill=np.nan):
    out = np.full(length, fill, dtype=np.float32)
    if len(arr) > 0:
        out[: min(len(arr), length)] = arr[:length]
    return out


def build_metrics_history(metrics):
    policy_loss, policy_key = pick_series(metrics, POLICY_LOSS_KEYS)
    value_loss, value_key = pick_series(metrics, VALUE_LOSS_KEYS)
    entropy, entropy_key = pick_series(metrics, ENTROPY_KEYS)
    bellman, bellman_key = pick_series(metrics, BELLMAN_KEYS)
    episode_return, return_key = pick_series(metrics, RETURN_KEYS)
    episode_length, ep_length_key = pick_series(metrics, EP_LENGTH_KEYS)

    train_len = max(len(policy_loss), len(value_loss), len(entropy), len(bellman), 1)
    policy_loss = pad_to_length(policy_loss, train_len)
    value_loss = pad_to_length(value_loss, train_len)

    if len(entropy) == 0:
        entropy = np.full(train_len, ENTROPY_FALLBACK_VALUE, dtype=np.float32)
    else:
        entropy = pad_to_length(entropy, train_len)

    if len(bellman) == 0 and BELLMAN_FALLBACK_TO_VALUE_LOSS:
        bellman = value_loss.copy()
    else:
        bellman = pad_to_length(bellman, train_len)

    if len(episode_length) == 0:
        episode_length = np.full(len(episode_return), float(MAX_CYCLES), dtype=np.float32)

    ep_len = max(len(episode_return), 1)
    episode_return = pad_to_length(episode_return, ep_len)
    episode_length = pad_to_length(episode_length, ep_len, fill=float(MAX_CYCLES))
    episode_rewards = episode_return / np.maximum(episode_length, 1e-8)

    history = {
        "iterations": list(range(1, train_len + 1)),
        "episode_iterations": list(range(1, ep_len + 1)),
        "policy_loss": policy_loss,
        "value_loss": value_loss,
        "entropy": entropy,
        "mean_bellman_error": bellman,
        "mean_episode_return": episode_return,
        "mean_episode_rewards": episode_rewards,
        "mean_episode_reward": episode_rewards,
    }

    chosen_keys = {
        "policy_loss": policy_key,
        "value_loss": value_key,
        "entropy": entropy_key,
        "mean_bellman_error": bellman_key,
        "mean_episode_return": return_key,
        "mean_episode_rewards": return_key,
        "mean_episode_length": ep_length_key,
    }
    return history, chosen_keys


def plot_metrics(metrics_history, save_path):
    plt.style.use("ggplot")
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))

    iteration_x = metrics_history["iterations"]
    episode_iteration_x = metrics_history["episode_iterations"]

    axes[0, 0].plot(iteration_x, metrics_history["policy_loss"])
    axes[0, 0].set_title("policy_loss")
    axes[0, 0].set_xlabel("Iteration")
    axes[0, 0].set_ylabel("policy_loss")
    axes[0, 0].grid(True, alpha=0.4)

    axes[0, 1].plot(iteration_x, metrics_history["value_loss"])
    axes[0, 1].set_title("value_loss")
    axes[0, 1].set_xlabel("Iteration")
    axes[0, 1].set_ylabel("value_loss")
    axes[0, 1].grid(True, alpha=0.4)

    axes[1, 0].plot(iteration_x, metrics_history["entropy"])
    axes[1, 0].set_title("entropy")
    axes[1, 0].set_xlabel("Iteration")
    axes[1, 0].set_ylabel("entropy")
    axes[1, 0].grid(True, alpha=0.4)

    axes[1, 1].plot(iteration_x, metrics_history["mean_bellman_error"])
    axes[1, 1].set_title("mean_bellman_error")
    axes[1, 1].set_xlabel("Iteration")
    axes[1, 1].set_ylabel("mean_bellman_error")
    axes[1, 1].grid(True, alpha=0.4)

    axes[2, 0].plot(episode_iteration_x, metrics_history["mean_episode_return"])
    axes[2, 0].set_title("mean_episode_return")
    axes[2, 0].set_xlabel("Iteration")
    axes[2, 0].set_ylabel("mean_episode_return")
    axes[2, 0].grid(True, alpha=0.4)

    axes[2, 1].plot(episode_iteration_x, metrics_history["mean_episode_rewards"])
    axes[2, 1].set_title("mean_episode_rewards")
    axes[2, 1].set_xlabel("Iteration")
    axes[2, 1].set_ylabel("mean_episode_rewards")
    axes[2, 1].grid(True, alpha=0.4)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Metrics plot saved to {save_path}")


In [ ]:
print(f"Device: {VMAS_DEVICE} | use_cuda={USE_CUDA}")
print(f"Output dir: {OUTPUT_DIR}")

run_dir = run_epymarl_experiment()
metrics = load_sacred_metrics(run_dir)
metrics_history, chosen_keys = build_metrics_history(metrics)

print("Sacred run:", run_dir)
print("Metric key mapping:", chosen_keys)
plot_metrics(metrics_history, PLOT_PATH)
